# Path Patching

**Standard activation patching** tests whether a component is important by replacing its *entire output*. But this conflates all the different roles a component might play. Head H writes a vector to the residual stream, and that vector is read by every downstream head and MLP. Head `H` might send critical information to head `K` (say, the identity of a duplicated name) while simultaneously sending irrelevant information to head `J` (say, positional encoding noise). Standard activation patching cannot distinguish these pathways - it can only tell you that `H` matters overall, not where its output is consumed.

**Path patching** asks a more targeted question: is the specific connection from `H` to `K` important? Instead of replacing H's entire output in the residual stream, path patching replaces *only the component* of `H`'s output that flows into a specific downstream consumer `K`.

The conceptual shift is from **nodes** to **edges** in the computational graph:

- **Activation patching tests nodes:** "Is component `H` important?"
- **Path patching tests edges:** "Is the connection `H→K` important?"

# ACDC - Automated Circuit DisCovery

ACDC starts with the full computational graph of the model, treating every possible edge between components as a candidate connection. It then iteratively tests each edge using path patching: if removing an edge has negligible effect on model behavior for the task, that edge is pruned. After testing all edges, what remains is the circuit - the **minimal subgraph** that accounts for the model's behavior.

The algorithm proceeds in topological order, working **backward from the output**:

1. Start with all edges in the computational graph
2. For each edge (in reverse topological order), temporarily remove it
3. If the model's behavior on the task is unchanged, permanently prune the edge
4. If behavior degrades, keep the edge
5. The surviving edges define the circuit

The threshold for "unchanged" is a tunable parameter, creating a tradeoff between faithfulness (keeping all edges that matter) and minimality (removing as many as possible). A strict threshold keeps more edges and produces a more faithful but less interpretable circuit. A loose threshold prunes aggressively and produces a more minimal but potentially less faithful circuit.

# Combining the Tools

Attribution patching and path patching are not alternatives to activation patching - they extend it. A typical circuit discovery workflow uses all three tools at different stages:

1. Attribution patching for broad screening. Sweep the entire model to identify which components show the largest estimated patching effects. This narrows the search from thousands of components to a manageable set of candidates.
2. Activation patching for confirmation. Run full patching on the top candidates to verify that the gradient approximation was accurate. This catches components where the linear approximation was misleading.
3. Path patching for mechanistic understanding. Once the key components are identified, trace the connections between them. This reveals not just which components participate in the circuit but how information flows between them.

The progression moves from "something is happening at layer 9" (attribution patching) to "head 9.9 is causally important" (activation patching) to "head 9.9 receives S-Inhibition information through its queries and copies name identity through its OV circuit to the output logits" (path patching). Each step adds resolution and mechanistic detail.

In [1]:
from nnsight import VisionLanguageModel
from PIL import Image
import json
import os
from pathlib import Path
import torch
import numpy as np
from transformers import LlavaForConditionalGeneration, AutoProcessor
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import einops
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected+notebook"

In [2]:
NB_DIR = Path(os.getcwd())  # patching/
PROJECT_DIR = NB_DIR.parent

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
POPE_FILE = PROJECT_DIR / "pope_train" / "coco_train_pope_adversarial.jsonl"
IMAGE_ROOT = PROJECT_DIR / "data" / "coco2014" / "train2014" / "train2014"

NUM_HEADS = 32
NUM_LAYERS = 32
HIDDEN_DIM = 4096
HEAD_DIM = HIDDEN_DIM // NUM_HEADS  # 128

In [3]:
model = VisionLanguageModel(
    "llava-hf/llava-1.5-7b-hf",
    device_map="cuda",
    dispatch=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

lm = model.model.language_model
print(f"Model loaded on {model.device}")
print(f"Language model layers: {len(lm.layers)}")
print(f"Hidden dim: {lm.config.hidden_size}")
print(f"Num attention heads: {lm.config.num_attention_heads}")

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Model loaded on cuda:0
Language model layers: 32
Hidden dim: 4096
Num attention heads: 32


In [4]:
# Get token IDs for "Yes" and "No" — used for probability difference p(Yes) - p(No)
yes_id = tokenizer.encode("Yes", add_special_tokens=False)[0]
no_id = tokenizer.encode("No", add_special_tokens=False)[0]
print(f"'Yes' token id: {yes_id}, '{tokenizer.decode([yes_id])}'")
print(f"'No'  token id: {no_id}, '{tokenizer.decode([no_id])}'")

'Yes' token id: 3869, 'Yes'
'No'  token id: 1939, 'No'


## 0. Helpers

In [5]:
def build_prompt(question_text: str) -> str:
    """Build the LLaVA prompt exactly as in the POPE data — no extra instructions."""
    return (
        f"USER: <image>\n{question_text}\nASSISTANT:"
    )

def get_prob_diff(logits):
    """Extract p(Yes) - p(No) from model outputs using softmax over the vocabulary.
    Positive → model prefers Yes. Negative → model prefers No."""
    logits = logits[0, -1, :]  # last token position
    probs = torch.softmax(logits.float(), dim=-1)
    return probs[yes_id].item() - probs[no_id]

def add_gaussian_noise(image: Image.Image, mean: float = 0.0, std: float = 25.5) -> Image.Image:
    """
    Adds Gaussian noise to a PIL Image.
    """
    # Convert image to numpy array (float32 to prevent overflow during addition)
    img_array = np.array(image).astype(np.float32)
    
    # Generate Gaussian noise
    noise = np.random.normal(mean, std, img_array.shape)
    
    # Add noise, clip to valid pixel range [0, 255], and convert back to uint8
    noisy_img_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    
    return Image.fromarray(noisy_img_array)

def prepare_inputs(prompt: str, image, processor, noise_std, device):
    """
    Prepares clean and corrupted inputs for LLaVA 1.5.
    """
    # 1. Create the corrupted version of the image
    corrupted_image = add_gaussian_noise(image, std=noise_std)
    
    # 2. Process the clean inputs
    clean_inputs = processor(
        text=prompt, 
        images=image, 
        return_tensors="pt"
    ).to(device)
    
    # 3. Process the corrupted inputs
    corrupted_inputs = processor(
        text=prompt, 
        images=corrupted_image, 
        return_tensors="pt"
    ).to(device)
    
    return clean_inputs, corrupted_inputs

def plot_heatmap(effects, baseline_pd, title, vmin=None, vmax=None):
    """Plot a single 32×32 heatmap of attribution scores (gradient × activation)."""
    if vmin is None:
        vmax_abs = max(abs(effects.min()), abs(effects.max()))
        vmin, vmax = -vmax_abs, vmax_abs

    fig, ax = plt.subplots(1, 1, figsize=(8, 7))
    sns.heatmap(
        effects,
        ax=ax,
        cmap="RdBu_r",
        center=0,
        vmin=vmin,
        vmax=vmax,
        xticklabels=range(0, 32),
        yticklabels=range(0, 32),
        cbar_kws={"label": "Attribution Score  ∇[p(Yes)−p(No)] · activation  (grad × act)", "shrink": 0.8},
        linewidths=0.0,
    )
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_title(f"{title}\n(baseline p(Yes)−p(No) = {baseline_pd:.4f})")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


In [6]:
# Load existing model answers and join with labels to classify TP/TN/FP
ANS_FILE = PROJECT_DIR / "results" / "train" / "llava15_7b_adversarial_train.jsonl"
LABEL_FILE = PROJECT_DIR / "pope_train" / "coco_train_pope_adversarial.jsonl"

answers = []
with open(ANS_FILE) as f:
    for line in f:
        if line.strip():
            answers.append(json.loads(line))

labels = []
with open(LABEL_FILE) as f:
    for line in f:
        if line.strip():
            labels.append(json.loads(line))

# Index labels by question_id
label_by_qid = {l["question_id"]: l for l in labels}

results = []
for ans in answers:
    qid = ans["question_id"]
    lab = label_by_qid.get(qid)
    if lab is None:
        continue
    pred = ans["answer"]
    results.append((lab, pred, None))

print(f"Loaded {len(results)} matched (answer + label) pairs")

Loaded 10002 matched (answer + label) pairs


In [7]:
# Classify into TP, TN, FP, FN using saved answers (no logit_diff yet)
tp = [(item, None) for item, pred, _ in results if item["label"] == "yes" and pred == "yes"]
tn = [(item, None) for item, pred, _ in results if item["label"] == "no"  and pred == "no"]
fp = [(item, None) for item, pred, _ in results if item["label"] == "no"  and pred == "yes"]
fn = [(item, None) for item, pred, _ in results if item["label"] == "yes" and pred == "no"]

print(f"TP: {len(tp)} | TN: {len(tn)} | FP: {len(fp)} | FN: {len(fn)}")

# Pick one representative of each type
tp_example, _ = tp[0]
tn_example, _ = tn[0]
fp_example, _ = fp[0]

print(f"\nTP example: {tp_example['text'][:80]} (label={tp_example['label']})")
print(f"TN example: {tn_example['text'][:80]} (label={tn_example['label']})")
print(f"FP example: {fp_example['text'][:80]} (label={fp_example['label']})")

TP: 4654 | TN: 3333 | FP: 1668 | FN: 347

TP example: Is there a bottle in the image? (label=yes)
TN example: Is there a car in the image? (label=no)
FP example: Is there a bowl in the image? (label=no)


## 1. Attribution Patching

The core insight of attribution patching is that we can approximate the effect of patching each component without actually performing the patch. The idea relies on a first-order Taylor approximation: if we know how sensitive the output metric is to changes at each activation (the gradient), and we know how much each activation changes between clean and corrupted runs (the activation difference), we can estimate the patching effect as their product.¹

Formally, the estimated patching effect for activation $a_i$ is:

$$\text{Patch effect of } a_i \approx \nabla_{a_i}\mathcal{L} \cdot (a_i^{\text{clean}} - a_i^{\text{corrupt}})$$

The gradient $\nabla_{a_i}\mathcal{L}$ captures the local sensitivity of the metric to perturbations at $a_i$. The difference $(a_i^{\text{clean}} - a_i^{\text{corrupt}})$ captures how much the activation actually changes between the two runs. Their dot product estimates how much the metric would change if we replaced the corrupted activation with the clean one at that location.

The efficiency gain is dramatic. Full activation patching requires $O(n)$ forward passes, where $n$ is the number of components. Attribution patching requires exactly two forward passes (one clean, one corrupted) plus one backward pass (to compute gradients). That is three passes total, regardless of model size. For GPT-3 with 4.7 million neurons, this means 3 passes instead of 4.7 million.

In [8]:
def attribution_patching(model, clean_inputs, corrupted_inputs):
    clean_out = []
    corrupted_out = []
    corrupted_grads = []

    # Pass 1  clean run.
    with model.trace(**clean_inputs):
        for layer in model.model.language_model.layers:
            # Clean attention output for this layer, across all heads
            attn_out = layer.self_attn.o_proj.output
            clean_out.append(attn_out.save())

        outputs = model.lm_head.output.save()
        baseline_prob_diff = get_prob_diff(outputs.cpu()).save()

    # Pass 2 corrupted run + backward.
    with model.trace(**corrupted_inputs):
        corrupted_refs = []
        for layer in model.model.language_model.layers:
            # Corrupted attention output for this layer, across all heads
            attn_out = layer.self_attn.o_proj.output
            attn_out.requires_grad_(True)        # non-leaf tensor needs this for a grad
            corrupted_refs.append(attn_out)
            corrupted_out.append(attn_out.save())

        outputs = model.lm_head.output.save()

        value = get_prob_diff(outputs.cpu())

        with value.backward():
            for attn_out in reversed(corrupted_refs):
                corrupted_grads.insert(0, attn_out.grad.save())

    patching_results = []

    for corrupted_grad, corrupted, clean, layer in zip(
        corrupted_grads, corrupted_out, clean_out, range(len(clean_out))
    ):

        residual_attr = einops.reduce(
            corrupted_grad[:,-1,:] * (clean[:,-1,:] - corrupted[:,-1,:]),
            "batch (head dim) -> head",
            "sum",
            head = 32,
            dim = 128,
        )

        patching_results.append(
            residual_attr.detach().cpu().numpy()
        )

    model.zero_grad()
    del corrupted_grads, corrupted_out, clean_out
    torch.cuda.empty_cache()

    return baseline_prob_diff, patching_results

In [9]:
def print_top_heads(effects, label, n=8):
    """Print the top-N most influential heads (by absolute effect)."""
    flat = [(effects[l, h], l, h) for l in range(NUM_LAYERS) for h in range(NUM_HEADS)]
    flat.sort(key=lambda x: -abs(x[0]))
    print(f"\n{label} — top {n} most influential heads:")
    for i, (val, l, h) in enumerate(flat[:n]):
        direction = "promotes pred" if val > 0 else "suppresses pred"
        print(f"  {i+1}. L{l:02d}H{h:02d}  indirect_effect={val:+.3f}  ({direction})")

In [10]:
N_SAMPLES = 500

# Pick N_SAMPLES from each category (skip if not enough)
import random
random.seed(42)

tp_samples = random.sample(tp, min(N_SAMPLES, len(tp)))
tn_samples = random.sample(tn, min(N_SAMPLES, len(tn)))
fp_samples = random.sample(fp, min(N_SAMPLES, len(fp)))

print(f"Selected {len(tp_samples)} TP, {len(tn_samples)} TN, {len(fp_samples)} FP")

# Extend inputs_cache with all new samples
all_samples = [("TP", s[0]) for s in tp_samples] + \
              [("TN", s[0]) for s in tn_samples] + \
              [("FP", s[0]) for s in fp_samples]

def attribution_patching_multi(samples, label, model, processor, noise_std=25.5):
    """
    Run attribution patching on multiple samples and return the results.
    """
    baselines = []
    all_effects = np.zeros((len(samples), NUM_LAYERS, NUM_HEADS))

    for idx, (item, _) in enumerate(tqdm(samples, desc=f"Attributing {label}")):
        baseline_pd, effects = attribution_patching(model, *prepare_inputs(build_prompt(item["text"]), Image.open(IMAGE_ROOT / item["image"]).convert("RGB"), processor, noise_std, model.device))
        baselines.append(baseline_pd)
        all_effects[idx] = effects

    return baselines, all_effects


Selected 500 TP, 500 TN, 500 FP


In [11]:
tp_baselines, tp_all_effects = attribution_patching_multi(tp_samples, "TP", model, processor)
tn_baselines, tn_all_effects = attribution_patching_multi(tn_samples, "TN", model, processor)
fp_baselines, fp_all_effects = attribution_patching_multi(fp_samples, "FP", model, processor)

Attributing TP:   1%|          | 4/500 [00:05<11:46,  1.42s/it]
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3748, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_42190/1797773026.py", line 1, in <module>
    tp_baselines, tp_all_effects = attribution_patching_multi(tp_samples, "TP", model, processor)
  File "/tmp/ipykernel_42190/3435352100.py", line 26, in attribution_patching_multi
    baseline_pd, effects = attribution_patching(model, *prepare_inputs(build_prompt(item["text"]), Image.open(IMAGE_ROOT / item["image"]).convert("RGB"), processor, noise_std, model.device))
  File "/tmp/ipykernel_42190/451513819.py", line 17, in attribution_patching
    with model.trace(**corrupted_inputs):
  File "/venv/main/lib/python3.12/site-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "/venv/main/lib/python3.12/site-packages/torch/autograd/__i

In [ ]:
tp_samples

[({'question_id': 1971,
   'image': 'COCO_train2014_000000166920.jpg',
   'text': 'Is there a person in the image?',
   'label': 'yes'},
  None),
 ({'question_id': 449,
   'image': 'COCO_train2014_000000350514.jpg',
   'text': 'Is there a suitcase in the image?',
   'label': 'yes'},
  None),
 ({'question_id': 4849,
   'image': 'COCO_train2014_000000320707.jpg',
   'text': 'Is there an orange in the image?',
   'label': 'yes'},
  None),
 ({'question_id': 4317,
   'image': 'COCO_train2014_000000515216.jpg',
   'text': 'Is there a person in the image?',
   'label': 'yes'},
  None),
 ({'question_id': 3927,
   'image': 'COCO_train2014_000000246009.jpg',
   'text': 'Is there a potted plant in the image?',
   'label': 'yes'},
  None),
 ({'question_id': 2465,
   'image': 'COCO_train2014_000000385505.jpg',
   'text': 'Is there a cell phone in the image?',
   'label': 'yes'},
  None),
 ({'question_id': 1821,
   'image': 'COCO_train2014_000000425550.jpg',
   'text': 'Is there a fork in the image?